# Sensor event extraction

Can a first-month baseline convert dense sensor observations into temporal events?

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.data import write_json

pd.set_option("display.max_columns", 20)

from src.sensor_events import fit_robust_baseline, score_sensor_points, aggregate_windows, merge_event_windows

sensor = pd.read_parquet(ROOT / "data/processed/metropt3.parquet")
out = ROOT / "results/sensor"
out.mkdir(parents=True, exist_ok=True)

## Calibration

UCI recommends the first month for training. We use February 2020 and condition on COMP; this is an assumed reference period, not a certified fault-free sample. Quantiles and aggregation settings are fixed in config before failure-window validation.

In [2]:
baseline = sensor.loc[sensor["timestamp"].ge(config.BASELINE_START) & sensor["timestamp"].lt(config.BASELINE_END)]
assert not baseline.empty
stats = fit_robust_baseline(baseline)
baseline_points, baseline_scores = score_sensor_points(baseline, stats)
baseline_windows = aggregate_windows(baseline_points, baseline_scores)
nonempty = baseline_windows.loc[baseline_windows["n_obs"].gt(0)]
minimum_observations = int(nonempty["n_obs"].quantile(.05))
calibration_windows = nonempty.loc[nonempty["n_obs"].ge(minimum_observations)]
window_threshold = float(calibration_windows["score"].quantile(config.WINDOW_THRESHOLD_QUANTILE))
stats.to_csv(out / "baseline_stats.csv", index=False)
calibration = {
    "baseline_start": config.BASELINE_START, "baseline_end": config.BASELINE_END,
    "baseline_rows": len(baseline), "operating_regime": "COMP", "sensor_quantile": config.SENSOR_QUANTILE,
    "window_freq": config.WINDOW_FREQ, "window_score_quantile": config.WINDOW_SCORE_QUANTILE,
    "window_threshold_quantile": config.WINDOW_THRESHOLD_QUANTILE, "window_threshold": window_threshold,
    "minimum_observations": minimum_observations, "coverage_quantile": .05,
    "merge_gap_minutes": config.MERGE_GAP_MINUTES,
    "constant_channel_rule": "binary novelty score 2 for a change from a constant baseline"
}
write_json(out / "calibration.json", calibration)
display(stats)
display(pd.DataFrame([calibration]))

,COMP,sensor,n,median,mad,iqr,std,scale,scale_method,abs_z_threshold,quantile
0,0.0,TP2,13031,9.3920,0.5820,1.178,1.818422,0.862873,MAD,10.900791,0.999
1,0.0,TP3,13031,9.1060,0.5720,1.156,0.659824,0.848047,MAD,1.789995,0.999
2,0.0,H1,13031,-0.0140,0.0040,0.008,0.372728,0.005930,MAD,1249.774046,0.999
3,0.0,DV_pressure,13031,-0.0180,0.0020,0.004,0.092633,0.002965,MAD,3.372454,0.999
4,0.0,Reservoirs,13031,9.1060,0.5720,1.154,0.658781,0.848047,MAD,1.798862,0.999
5,0.0,Oil_temperature,13031,55.1750,1.9250,3.850,3.503123,2.854005,MAD,6.113952,0.999
6,0.0,Motor_current,13031,6.0050,0.1250,0.255,0.324972,0.185325,MAD,8.250506,0.999
7,1.0,TP2,201819,-0.0120,0.0020,0.002,0.185551,0.002965,MAD,77.811952,0.999
8,1.0,TP3,201819,8.9280,0.4540,0.912,0.543775,0.673100,MAD,1.770910,0.999
9,1.0,H1,201819,8.9120,0.4540,0.910,0.553054,0.673100,MAD,1.791709,0.999


,baseline_start,baseline_end,baseline_rows,operating_regime,sensor_quantile,window_freq,window_score_quantile,window_threshold_quantile,window_threshold,minimum_observations,coverage_quantile,merge_gap_minutes,constant_channel_rule
0,2020-02-01,2020-03-01,214850,COMP,0.999,5min,0.9,0.995,0.993238,30,0.05,10,binary novelty score 2 for a change from a con...


## Scoring and temporal aggregation

In [3]:
points, scores = score_sensor_points(sensor, stats)
windows = aggregate_windows(points, scores)
windows["eligible"] = windows["n_obs"].ge(minimum_observations) & (windows.index >= config.BASELINE_END)
windows["candidate"] = windows["eligible"] & windows["score"].gt(window_threshold)
events = merge_event_windows(windows)
windows.reset_index().to_parquet(out / "window_scores.parquet", index=False)
events.to_csv(out / "events.csv", index=False)
display(events.head(10))
display(windows[["n_obs", "score", "eligible", "candidate"]].describe())

,event_id,start,end,duration_min,n_windows,score_max,peak_sensor,peak_sensor_score,active_sensors
0,SE_00001,2020-03-01 05:30:00,2020-03-01 05:35:00,5.0,1,1.000000,DV_pressure,21.017799,TP2|TP3|H1|DV_pressure|Reservoirs|Motor_current
1,SE_00002,2020-03-01 05:50:00,2020-03-01 05:55:00,5.0,1,1.000000,H1,1.076285,TP3|H1|Reservoirs
2,SE_00003,2020-03-01 06:15:00,2020-03-01 06:55:00,40.0,4,1.017369,H1,1.063018,TP3|H1|Reservoirs|Oil_temperature
3,SE_00004,2020-03-01 07:10:00,2020-03-01 07:15:00,5.0,1,1.018550,H1,1.061360,TP3|H1|Reservoirs|Oil_temperature
4,SE_00005,2020-03-01 07:30:00,2020-03-01 07:35:00,5.0,1,1.059256,DV_pressure,350.800000,TP3|H1|DV_pressure|Reservoirs|Oil_temperature
5,SE_00006,2020-03-01 07:50:00,2020-03-01 07:55:00,5.0,1,1.027103,H1,1.067993,TP3|H1|Reservoirs|Oil_temperature
6,SE_00007,2020-03-01 08:10:00,2020-03-01 08:15:00,5.0,1,1.042664,H1,1.069652,TP3|H1|Reservoirs|Oil_temperature
7,SE_00008,2020-03-01 08:30:00,2020-03-01 08:35:00,5.0,1,1.035919,H1,1.074627,TP3|H1|Reservoirs|Oil_temperature
8,SE_00009,2020-03-01 08:50:00,2020-03-01 08:55:00,5.0,1,1.022597,H1,1.058043,TP3|H1|Reservoirs|Oil_temperature
9,SE_00010,2020-03-01 09:10:00,2020-03-01 09:15:00,5.0,1,1.059528,TP2,3.501959,TP2|TP3|H1|DV_pressure|Reservoirs|Oil_temperature


,n_obs,score
count,61392.000000,50782.000000
mean,24.709213,8.558695
std,11.459782,37.736886
min,0.000000,0.052277
25%,30.000000,0.555872
50%,30.000000,0.917184
75%,30.000000,0.975186
max,31.000000,334.800000


## Retrospective validation

The detector and events are now fixed. Official failure windows are used only to measure temporal overlap; unreported intervals are not reliable negative labels. Broad event coverage can make overlap easy and must be reported alongside it.

In [4]:
failure_windows = pd.DataFrame([
    {"failure_id": "F1", "start": "2020-04-18 00:00", "end": "2020-04-18 23:59"},
    {"failure_id": "F2", "start": "2020-05-29 23:30", "end": "2020-05-30 06:00"},
    {"failure_id": "F3", "start": "2020-06-05 10:00", "end": "2020-06-07 14:30"},
    {"failure_id": "F4", "start": "2020-07-15 14:30", "end": "2020-07-15 19:00"}
])
failure_windows[["start", "end"]] = failure_windows[["start", "end"]].apply(pd.to_datetime)
validation = []
for row in failure_windows.itertuples(index=False):
    overlap = events.loc[events["start"].lt(row.end) & events["end"].gt(row.start)]
    observed = windows.loc[(windows.index >= row.start) & (windows.index < row.end) & windows["eligible"]]
    validation.append({"failure_id": row.failure_id, "start": str(row.start), "end": str(row.end),
                       "overlapping_events": len(overlap), "detected_during_failure": not overlap.empty,
                       "observed_windows": len(observed), "candidate_windows": int(observed["candidate"].sum())})
validation = pd.DataFrame(validation)
validation.to_csv(out / "failure_overlap.csv", index=False)
eligible = windows.loc[windows["eligible"]]
summary = {
    "calibration": calibration, "events": len(events), "candidate_windows": int(windows["candidate"].sum()),
    "eligible_windows": len(eligible), "candidate_fraction": float(eligible["candidate"].mean()),
    "unobserved_or_sparse_windows": int((~windows["n_obs"].ge(minimum_observations)).sum()),
    "event_span_hours": float(events["duration_min"].sum() / 60),
    "eventized_observed_hours": float(windows["candidate"].sum() * pd.Timedelta(config.WINDOW_FREQ).total_seconds() / 3600),
    "failures_overlapped": int(validation["detected_during_failure"].sum()), "failure_windows": len(validation),
    "failure_source": config.METROPT_SOURCE,
    "interpretation": "high-sensitivity eventization; not a validated final anomaly detector"
}
write_json(out / "summary.json", summary)
display(validation)
display(pd.DataFrame([{k: v for k, v in summary.items() if k != "calibration"}]))

,failure_id,start,end,overlapping_events,detected_during_failure,observed_windows,candidate_windows
0,F1,2020-04-18 00:00:00,2020-04-18 23:59:00,1,True,283,283
1,F2,2020-05-29 23:30:00,2020-05-30 06:00:00,1,True,78,78
2,F3,2020-06-05 10:00:00,2020-06-07 14:30:00,3,True,569,569
3,F4,2020-07-15 14:30:00,2020-07-15 19:00:00,1,True,53,53


,events,candidate_windows,eligible_windows,candidate_fraction,unobserved_or_sparse_windows,event_span_hours,eventized_observed_hours,failures_overlapped,failure_windows,failure_source,interpretation
0,2934,9472,41042,0.230788,13283,1110.333333,789.333333,4,4,https://archive.ics.uci.edu/dataset/791/metrop...,high-sensitivity eventization; not a validated...
